<a href="https://colab.research.google.com/github/elvisvaquerano/machine-learning-assignment/blob/main/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Generate sample customer data


# Load real customer churn data
# Data Source: Kaggle - Telco Customer Churn
# IBM Telco Customer Churn Dataset
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Remove rows with missing values
df = df.dropna()

# Convert Churn from Yes/No to 1/0
df['churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Select real customer features
X = df[['tenure', 'MonthlyCharges', 'TotalCharges',
        'Contract', 'InternetService']]

# Target variable
y = df['churn']

# Display dataset information
print("Number of customers:", len(df))
print(df[['tenure', 'MonthlyCharges', 'TotalCharges',
          'Contract', 'InternetService', 'churn']].head())


# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),
         ['tenure', 'MonthlyCharges', 'TotalCharges']),

        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         ['Contract', 'InternetService'])
    ])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'tenure': [6],
    'MonthlyCharges': [85.00],
    'TotalCharges': [510.00],
    'Contract': ['Month-to-month'],
    'InternetService': ['Fiber optic']
})
churn_probability = model.predict_proba(new_customer)[0][1]
# Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients

# Get categorical feature names
categorical_names = (
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(['Contract', 'InternetService'])
    .tolist()
)

# Numerical feature names
numerical_names = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Combine feature names in the same order as preprocessing
feature_names = numerical_names + categorical_names

# Get logistic regression coefficients
coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

# Evaluate model performance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Evaluation:")
print(f"Accuracy: {accuracy:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Number of customers: 7032
   tenure  MonthlyCharges  TotalCharges        Contract InternetService  churn
0       1           29.85         29.85  Month-to-month             DSL      0
1      34           56.95       1889.50        One year             DSL      0
2       2           53.85        108.15  Month-to-month             DSL      1
3      45           42.30       1840.75        One year             DSL      0
4       2           70.70        151.65  Month-to-month     Fiber optic      1
Churn Probability for new customer: 0.66
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
tenure: -1.43
MonthlyCharges: -0.05
TotalCharges: 0.73
Contract_Month-to-month: 0.90
Contract_One year: -0.07
Contract_Two year: -0.84
InternetService_DSL: -0.03
InternetService_Fiber optic: 1.04
InternetService_No: -1.00

Model Evaluation:
Accuracy: 0.785

Confusion Matrix:
[[925 108]
 [194 180]]

Classification Report:
              precision    recall  f1-score   support

           0  

Conclusion:
This model used 7,032 customer records from the Telco Customer Churn dataset to predict whether a customer is likely to churn. The logistic regression model achieved an accuracy of 78.5%. For the example customer, the model predicted a 66% probability of churn, which exceeded the 50% threshold and classified the customer as at risk of leaving. The results demonstrate how customer characteristics such as tenure, monthly charges, contract type, and internet service can be used to predict churn. However, the model was less successful at identifying customers who actually churned, so additional customer features or more advanced models could potentially improve performance.